# Week 2: Uninformed Search: 8-Puzzle

This week compares depth-first search (DFS) and iterative deepening search (IDS) on the same state-space problem. The deck also introduces breadth-first search as a contrast.

**Source:** `PPTs/Week2.pptx`.


### Theory note: BFS
Breadth-first search expands all states at depth *d* before states at depth *d+1* and finds a shortest path when step costs are equal. The practical exercises in this deck ask for DFS and IDS; BFS is summarized here for comparison.

## Program 1 — 8-Puzzle with Depth-First Search (DFS)

**Theory.** The 8-puzzle is a 3×3 board with eight numbered tiles and one blank. DFS explores one path deeply before backtracking. It can find a solution, but it does not guarantee the shortest path.

**Objective.** Find a path from a user-entered 8-puzzle state to the standard goal using DFS.

**Libraries.** Python standard library only.

**Algorithm.**
1. Validate the nine tiles. 2. Reject unsolvable permutations using inversion parity. 3. Put the start state on a stack. 4. Expand unseen neighbors and remember each parent. 5. Reconstruct the path on reaching the goal.

**Complexity.** Worst case O(b^m) time and O(b^m) visited-state space for this graph-search implementation (b ≤ 4, m is the explored depth).

The code below keeps `input()` so you can provide your own values. The final lines run a sample interaction through the same prompts; edit `SAMPLE_INPUTS` to try another case, or call the main function directly for a live terminal run.

**Expected output and explanation.** The output reports the computed result for the entered values.


In [1]:
from contextlib import contextmanager
import builtins

@contextmanager
def sample_input(values):
    original_input = builtins.input
    remaining = iter(map(str, values))
    def demo_input(prompt=''):
        print(prompt, end='')
        try:
            value = next(remaining)
        except StopIteration as exc:
            raise RuntimeError('Add more values to SAMPLE_INPUTS for this run.') from exc
        print(value)
        return value
    builtins.input = demo_input
    try:
        yield
    finally:
        builtins.input = original_input

from collections import deque

GOAL=(1,2,3,4,5,6,7,8,0)

def neighbors(state):
    z=state.index(0); r,c=divmod(z,3)
    for dr,dc in ((-1,0),(1,0),(0,-1),(0,1)):
        nr,nc=r+dr,c+dc
        if 0<=nr<3 and 0<=nc<3:
            j=nr*3+nc; child=list(state); child[z],child[j]=child[j],child[z]
            yield tuple(child)

def show(state):
    for i in (0,3,6): print(*state[i:i+3])

def valid_state(state):
    return len(state)==9 and set(state)==set(range(9))

def solvable(state):
    a=[x for x in state if x]; inv=sum(a[i]>a[j] for i in range(8) for j in range(i+1,8))
    return inv%2==0
def dfs(start,limit=50000):
    stack=[start]; parent={start:None}; expanded=0
    while stack:
        state=stack.pop()
        if state==GOAL: return reconstruct(parent,state),expanded
        expanded+=1
        if expanded>limit: return None,expanded
        for child in neighbors(state):
            if child not in parent:
                parent[child]=state; stack.append(child)
    return None,expanded

def reconstruct(parent,state):
    path=[]
    while state is not None: path.append(state); state=parent[state]
    return list(reversed(path))

def main():
    try: start=tuple(map(int,input('Enter 9 tiles, 0 for blank: ').split()))
    except ValueError: print('Use nine integers.'); return
    if not valid_state(start): print('Tiles must be each integer 0 through 8 exactly once.'); return
    if not solvable(start): print('This state cannot reach the standard goal.'); return
    try: limit=int(input('Maximum expanded states (positive integer): '))
    except ValueError: print('Limit must be an integer.'); return
    if limit<1: print('Limit must be positive.'); return
    path,expanded=dfs(start,limit)
    print('\nInitial state:'); show(start)
    if path is None: print(f'No path found within the search limit ({expanded} states expanded).'); return
    print(f'DFS path ({len(path)-1} moves; {expanded} states expanded):')
    for state in path: show(state); print()


SAMPLE_INPUTS = ['1 2 3 4 5 6 7 0 8', '50000']
with sample_input(SAMPLE_INPUTS):
    main()


Enter 9 tiles, 0 for blank: 1 2 3 4 5 6 7 0 8
Maximum expanded states (positive integer): 50000

Initial state:
1 2 3
4 5 6
7 0 8
DFS path (1 moves; 1 states expanded):
1 2 3
4 5 6
7 0 8

1 2 3
4 5 6
7 8 0



## Program 2 — 8-Puzzle with Iterative Deepening Search (IDS)

**Theory.** IDS repeatedly performs depth-limited DFS with increasing limits. It combines DFS-like memory use with breadth-first search’s guarantee of finding a shallowest solution when all moves have equal cost.

**Objective.** Solve the 8-puzzle within a user-chosen depth bound using IDS.

**Libraries.** Python standard library only.

**Algorithm.**
1. Validate solvability. 2. Run depth-limited DFS at limit 0. 3. Increase the limit and repeat. 4. Avoid states already on the current path. 5. Stop at the first goal or the user’s maximum depth.

**Complexity.** O(b^d) time and O(bd) recursion/path space (b ≤ 4, d is the solution depth).

The code below keeps `input()` so you can provide your own values. The final lines run a sample interaction through the same prompts; edit `SAMPLE_INPUTS` to try another case, or call the main function directly for a live terminal run.

**Expected output and explanation.** The output reports the computed result for the entered values.


In [2]:
from contextlib import contextmanager
import builtins

@contextmanager
def sample_input(values):
    original_input = builtins.input
    remaining = iter(map(str, values))
    def demo_input(prompt=''):
        print(prompt, end='')
        try:
            value = next(remaining)
        except StopIteration as exc:
            raise RuntimeError('Add more values to SAMPLE_INPUTS for this run.') from exc
        print(value)
        return value
    builtins.input = demo_input
    try:
        yield
    finally:
        builtins.input = original_input

from collections import deque

GOAL=(1,2,3,4,5,6,7,8,0)

def neighbors(state):
    z=state.index(0); r,c=divmod(z,3)
    for dr,dc in ((-1,0),(1,0),(0,-1),(0,1)):
        nr,nc=r+dr,c+dc
        if 0<=nr<3 and 0<=nc<3:
            j=nr*3+nc; child=list(state); child[z],child[j]=child[j],child[z]
            yield tuple(child)

def show(state):
    for i in (0,3,6): print(*state[i:i+3])

def valid_state(state):
    return len(state)==9 and set(state)==set(range(9))

def solvable(state):
    a=[x for x in state if x]; inv=sum(a[i]>a[j] for i in range(8) for j in range(i+1,8))
    return inv%2==0
def depth_limited(state,remaining,path,seen):
    if state==GOAL: return path[:]
    if remaining==0: return None
    for child in neighbors(state):
        if child not in seen:
            seen.add(child); result=depth_limited(child,remaining-1,path+[child],seen); seen.remove(child)
            if result is not None: return result
    return None

def ids(start,max_depth):
    total=0
    for depth in range(max_depth+1):
        print(f'Search depth: {depth}')
        result=depth_limited(start,depth,[start],{start})
        total+=1
        if result is not None: return result,total
    return None,total

def main():
    try: start=tuple(map(int,input('Enter 9 tiles, 0 for blank: ').split())); max_depth=int(input('Maximum depth to search: '))
    except ValueError: print('Enter nine integers and an integer depth.'); return
    if not valid_state(start): print('Tiles must be each integer 0 through 8 exactly once.'); return
    if max_depth<0: print('Depth must be nonnegative.'); return
    if not solvable(start): print('This state cannot reach the standard goal.'); return
    path,iterations=ids(start,max_depth)
    if path is None: print('No solution found within the depth limit.'); return
    print(f'Solution ({len(path)-1} moves):')
    for state in path: show(state); print()


SAMPLE_INPUTS = ['1 2 3 4 5 6 7 0 8', '10']
with sample_input(SAMPLE_INPUTS):
    main()


Enter 9 tiles, 0 for blank: 1 2 3 4 5 6 7 0 8
Maximum depth to search: 10
Search depth: 0
Search depth: 1
Solution (1 moves):
1 2 3
4 5 6
7 0 8

1 2 3
4 5 6
7 8 0

